# Notebook 5: Forecast Evaluation
**Project:** GARCH & Volatility Forecasting
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from arch import arch_model
import statsmodels.api as sm
from scipy import stats
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
spy = returns['SPY'] * 100
spy_raw = returns['SPY']
print(f'Loaded {len(spy):,} obs | {returns.index[0].date()} to {returns.index[-1].date()}')


Loaded 2,609 obs | 2015-01-01 to 2024-12-31


In [2]:
from forecast_evaluation import evaluate_all, diebold_mariano_test
import numpy as np

rv_actual=spy_raw**2
train_size=800; step=100
fc_garch,fc_egarch,fc_ewma,actuals=[],[],[],[]

for i in range(train_size,len(spy)-1,step):
    train=spy.iloc[:i]
    actual_next=rv_actual.iloc[i:i+step].values
    try:
        g=arch_model(train,vol='Garch',p=1,q=1,dist='Normal',mean='Constant').fit(disp='off')
        fc_garch.extend([g.forecast(horizon=1).variance.iloc[-1,0]/10000]*len(actual_next))
    except: fc_garch.extend([np.nan]*len(actual_next))
    try:
        e=arch_model(train,vol='EGARCH',p=1,q=1,dist='Normal',mean='Constant').fit(disp='off')
        fc_egarch.extend([e.forecast(horizon=1).variance.iloc[-1,0]/10000]*len(actual_next))
    except: fc_egarch.extend([np.nan]*len(actual_next))
    fc_ewma.extend([(train/100).ewm(span=63).var().iloc[-1]]*len(actual_next))
    actuals.extend(actual_next.tolist())

n=min(len(actuals),len(fc_garch),len(fc_egarch),len(fc_ewma))
actuals=np.array(actuals[:n])
forecasts={'GARCH(1,1)':np.array(fc_garch[:n]),'EGARCH':np.array(fc_egarch[:n]),'EWMA':np.array(fc_ewma[:n])}

results=evaluate_all(actuals,forecasts)
print('Walk-Forward Forecast Evaluation:')
print(results.round(6))
results.to_csv('../results/forecast_evaluation.csv')

/tmp/ipykernel_169/3177202388.py:16: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  e=arch_model(train,vol='EGARCH',p=1,q=1,dist='Normal',mean='Constant').fit(disp='off')
/tmp/ipykernel_169/3177202388.py:16: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  e=arch_model(train,vol='EGARCH',p=1,q=1,dist='Normal',mean='Constant').fit(disp='off')


Walk-Forward Forecast Evaluation:
            MSE      RMSE       MAE     QLIKE
model                                        
GARCH(1,1)  0.0  0.000467  0.000127 -7.917515
EGARCH      0.0  0.000468  0.000127 -7.904847
EWMA        0.0  0.000471  0.000135 -7.902400


In [3]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
models=results.index.tolist()
m_colors=['#185FA5','#E24B4A','#1D9E75']
ax1=axes[0]
mse_v=[results.loc[m,'MSE'] for m in models]
bars=ax1.bar(models,mse_v,color=m_colors,alpha=0.85,edgecolor='white')
for bar,v in zip(bars,mse_v):
    ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()*1.02,f'{v:.2e}',ha='center',va='bottom',fontsize=9)
ax1.set_title('MSE Loss (lower = better)'); ax1.set_ylabel('MSE')
ax2=axes[1]
qlike_v=[results.loc[m,'QLIKE'] for m in models]
bars2=ax2.bar(models,qlike_v,color=m_colors,alpha=0.85,edgecolor='white')
for bar,v in zip(bars2,qlike_v):
    ax2.text(bar.get_x()+bar.get_width()/2,bar.get_height()*1.001,f'{v:.4f}',ha='center',va='bottom',fontsize=9)
ax2.set_title('QLIKE Loss (lower = better)'); ax2.set_ylabel('QLIKE')
plt.tight_layout(); plt.savefig('../results/08_forecast_evaluation.png',dpi=150,bbox_inches='tight')
plt.show()
print(f'Best model by QLIKE: {results["QLIKE"].idxmin()}')

Best model by QLIKE: GARCH(1,1)
